In [10]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import soundfile as sf

from IPython.display import Audio, display, Markdown

# ============================================================
# CONFIGURACIÓN
# ============================================================

BASE_AUDIO = Path("vox1_test_wav/wav")
PROTOCOL_DIR = Path("protocolo_vox1")
RESULTS_DIR = Path("resultados")

N_CASES = 5

# Nombres EXACTOS de los modelos en puntajes.pkl
MODELOS = ["ECAPA", "WAVLM"]

print("Directorio de audios:", BASE_AUDIO.resolve())
print("Directorio de protocolos:", PROTOCOL_DIR.resolve())
print("Directorio de resultados:", RESULTS_DIR.resolve())

Directorio de audios: C:\Users\benja\OneDrive\Desktop\Universidad\Tareas\Voz\tarea3-verificacion-locutor\vox1_test_wav\wav
Directorio de protocolos: C:\Users\benja\OneDrive\Desktop\Universidad\Tareas\Voz\tarea3-verificacion-locutor\protocolo_vox1
Directorio de resultados: C:\Users\benja\OneDrive\Desktop\Universidad\Tareas\Voz\tarea3-verificacion-locutor\resultados


In [11]:

def get_problematic_indices(scores, labels, threshold, n=3):

    scores = np.asarray(scores)
    labels = np.asarray(labels)

    genuine_idx = np.where(labels == 1)[0]
    impostor_idx = np.where(labels == 0)[0]

    # Falsos rechazos: genuinos con menor similitud.
    false_rejections = genuine_idx[
        np.argsort(scores[genuine_idx])[:n]
    ]

    # Falsas aceptaciones: impostores con mayor similitud.
    false_acceptances = impostor_idx[
        np.argsort(scores[impostor_idx])[::-1][:n]
    ]

    # Comparaciones más cercanas al umbral EER.
    borderline = np.argsort(
        np.abs(scores - threshold)
    )[:n]

    return {
        "FR": false_rejections,
        "FA": false_acceptances,
        "Borderline": borderline
    }


def audio_path(rel_path):
    path = BASE_AUDIO / str(rel_path)

    if not path.exists():
        raise FileNotFoundError(
            f"No se encontró el audio: {path.resolve()}"
        )

    return path


def play_audio(rel_path, title=None):

    path = audio_path(rel_path)

    if title:
        display(Markdown(f"**{title}**"))

    display(Audio(filename=str(path)))


def show_case(model_name, protocol, case_type, idx, score):

    df = pairs if protocol == "A" else trials

    row = df.iloc[idx]

    label = int(row["label"])

    threshold = thresholds[f"{model_name} - {protocol}"]

    status = (
        "Genuino" if label == 1 else "Impostor"
    )

    prediction = (
        "Aceptado" if score >= threshold else "Rechazado"
    )

    display(Markdown(
        f"""
### {model_name} - Protocolo {protocol} - {case_type}

- **Índice:** {idx}
- **Etiqueta real:** {status}
- **Similitud coseno:** {score:.4f}
- **Umbral EER:** {threshold:.4f}
- **Decisión:** {prediction}
- **Distancia al umbral:** {abs(score-threshold):.4f}
"""
    ))

    if protocol == "A":

        audio1 = row["audio1"]
        audio2 = row["audio2"]

        play_audio(audio1, f"Audio 1: {audio1}")
        play_audio(audio2, f"Audio 2: {audio2}")

    else:

        speaker_id = row["enrollment_speaker_id"]
        test_audio = row["test_audio"]

        # Audios utilizados para construir el template.
        enrollment_audios = enrollment.loc[
            enrollment["speaker_id"] == speaker_id,
            "audio_path"
        ].tolist()

        display(Markdown(
            f"**Hablante de registro:** {speaker_id}"
        ))

        # Escuchar hasta dos audios del enrollment.
        for i, path in enumerate(enrollment_audios[:2], start=1):
            play_audio(
                path,
                f"Enrollment {i}: {path}"
            )

        play_audio(
            test_audio,
            f"Audio de prueba: {test_audio}"
        )


In [12]:
# ============================================================
# CARGAR PUNTAJES
# ============================================================

with open(RESULTS_DIR / "puntajes.pkl", "rb") as f:
    puntajes = pickle.load(f)

print("Puntajes disponibles:", list(puntajes.keys()))


# ============================================================
# PROTOCOLO A
# ============================================================

pairs = pd.read_csv(
    PROTOCOL_DIR / "veri_test_original.txt",
    sep=r"\s+",
    header=None,
    names=["label", "audio1", "audio2"]
)

pairs["speaker1"] = pairs["audio1"].apply(
    lambda x: Path(x).parts[0]
)

pairs["speaker2"] = pairs["audio2"].apply(
    lambda x: Path(x).parts[0]
)


# ============================================================
# PROTOCOLO B
# ============================================================

enrollment = pd.read_csv(
    PROTOCOL_DIR / "enrollment_10.csv"
)

probes = pd.read_csv(
    PROTOCOL_DIR / "probes.csv"
)

trials = pd.read_csv(
    PROTOCOL_DIR / "trials_registro10.csv"
)

# Normalizar identificadores
for col in ["speaker_id"]:
    enrollment[col] = enrollment[col].astype(str)

for col in ["enrollment_speaker_id", "test_speaker_id"]:
    trials[col] = trials[col].astype(str)

print("\nPares protocolo A:", len(pairs))
print("Audios de registro:", len(enrollment))
print("Audios de prueba:", len(probes))
print("Trials protocolo B:", len(trials))

Puntajes disponibles: ['ECAPA - A', 'ECAPA - B', 'WAVLM - A', 'WAVLM - B']

Pares protocolo A: 37720
Audios de registro: 400
Audios de prueba: 4315
Trials protocolo B: 8630


In [13]:
# ============================================================
# CARGAR UMBRALES
# ============================================================

metricas = pd.read_csv(RESULTS_DIR / "metricas.csv")

thresholds = {
    str(row["modelo_protocolo"]): float(row["umbral_EER"])
    for _, row in metricas.iterrows()
}

print("Umbrales:")
for key, value in thresholds.items():
    print(f"{key}: {value:.6f}")


# ============================================================
# VALIDAR ETIQUETAS DEL PROTOCOLO A
# ============================================================

pairs["label_correct"] = (
    (pairs["speaker1"] == pairs["speaker2"])
    == (pairs["label"] == 1)
)

# ============================================================
# VALIDAR ETIQUETAS DEL PROTOCOLO B
# ============================================================

trials["label_correct"] = (
    (trials["enrollment_speaker_id"] == trials["test_speaker_id"])
    == (trials["label"] == 1)
)

print("\n--- VALIDACIÓN PROTOCOLO A ---")
print("Total:", len(pairs))
print("Etiquetas inconsistentes:", (~pairs["label_correct"]).sum())

print("\n--- VALIDACIÓN PROTOCOLO B ---")
print("Total:", len(trials))
print("Etiquetas inconsistentes:", (~trials["label_correct"]).sum())


# ============================================================
# VALIDAR LONGITUD DE LOS PUNTAJES
# ============================================================

for model in MODELOS:
    for protocol, df in [("A", pairs), ("B", trials)]:

        key = f"{model} - {protocol}"

        if key not in puntajes:
            raise KeyError(f"No se encontró {key} en puntajes.pkl")

        if key not in thresholds:
            raise KeyError(f"No se encontró el umbral {key} en metricas.csv")

        scores = np.asarray(puntajes[key]).reshape(-1)

        print(
            f"{key}: {len(scores)} puntajes "
            f"(esperados: {len(df)})"
        )

        if len(scores) != len(df):
            raise ValueError(
                f"Cantidad de puntajes incorrecta para {key}"
            )

        if not np.all(np.isfinite(scores)):
            raise ValueError(f"Hay puntajes NaN o infinitos en {key}")


if not pairs["label_correct"].all():
    raise ValueError("Hay etiquetas inconsistentes en protocolo A.")

if not trials["label_correct"].all():
    raise ValueError("Hay etiquetas inconsistentes en protocolo B.")

print("\nTodas las validaciones superadas.")

Umbrales:
ECAPA - A: 0.272333
ECAPA - B: 0.371500
WAVLM - A: 0.873540
WAVLM - B: 0.908500

--- VALIDACIÓN PROTOCOLO A ---
Total: 37720
Etiquetas inconsistentes: 0

--- VALIDACIÓN PROTOCOLO B ---
Total: 8630
Etiquetas inconsistentes: 0
ECAPA - A: 37720 puntajes (esperados: 37720)
ECAPA - B: 8630 puntajes (esperados: 8630)
WAVLM - A: 37720 puntajes (esperados: 37720)
WAVLM - B: 8630 puntajes (esperados: 8630)

Todas las validaciones superadas.


In [14]:
def buscar_casos(scores, df, threshold, n=5):

    scores = np.asarray(scores).reshape(-1)
    labels = df["label"].to_numpy()

    # Falsos rechazos: genuinos clasificados como impostores
    fr_idx = np.where(
        (labels == 1) & (scores < threshold)
    )[0]

    fr_idx = fr_idx[np.argsort(scores[fr_idx])[:n]]

    # Falsas aceptaciones: impostores clasificados como genuinos
    fa_idx = np.where(
        (labels == 0) & (scores >= threshold)
    )[0]

    fa_idx = fa_idx[np.argsort(scores[fa_idx])[::-1]][:n]

    # Casos más cercanos al umbral
    borderline_idx = np.argsort(
        np.abs(scores - threshold)
    )[:n]

    seleccion = []

    for tipo, indices in [
        ("FR", fr_idx),
        ("FA", fa_idx),
        ("Borderline", borderline_idx)
    ]:

        for idx in indices:

            row = df.iloc[idx].to_dict()

            row.update({
                "indice": int(idx),
                "tipo": tipo,
                "score": float(scores[idx]),
                "threshold": float(threshold),
                "distancia_umbral": float(
                    abs(scores[idx] - threshold)
                ),
                "prediccion": int(scores[idx] >= threshold)
            })

            seleccion.append(row)

    return pd.DataFrame(seleccion)


# ============================================================
# GENERAR CASOS PARA TODOS LOS MODELOS Y PROTOCOLOS
# ============================================================

casos = {}

for model in MODELOS:

    for protocol, df in [
        ("A", pairs),
        ("B", trials)
    ]:

        key = f"{model} - {protocol}"

        scores = np.asarray(puntajes[key]).reshape(-1)
        threshold = thresholds[key]

        casos[key] = buscar_casos(
            scores=scores,
            df=df,
            threshold=threshold,
            n=N_CASES
        )

        print("\n" + "=" * 70)
        print(key)
        print("=" * 70)

        columnas = [
            "tipo",
            "indice",
            "label",
            "score",
            "threshold",
            "distancia_umbral",
            "prediccion"
        ]

        display(casos[key][columnas].reset_index(drop=True))


ECAPA - A


,tipo,indice,label,score,threshold,distancia_umbral,prediccion
0,FR,31156,1,-0.041027,0.272333,0.313360,0
1,FR,31176,1,-0.034864,0.272333,0.307197,0
2,FR,31178,1,-0.034650,0.272333,0.306984,0
3,FR,31182,1,-0.004190,0.272333,0.276523,0
4,FR,31584,1,-0.003068,0.272333,0.275402,0
5,FA,8441,0,0.477648,0.272333,0.205315,1
6,FA,28479,0,0.457135,0.272333,0.184801,1
7,FA,35759,0,0.433843,0.272333,0.161509,1
8,FA,8377,0,0.422620,0.272333,0.150287,1
9,FA,12039,0,0.413895,0.272333,0.141562,1



ECAPA - B


,tipo,indice,label,score,threshold,distancia_umbral,prediccion
0,FR,7110,1,0.009468,0.3715,0.362032,0
1,FR,7270,1,0.194421,0.3715,0.177079,0
2,FR,7280,1,0.209418,0.3715,0.162082,0
3,FR,7290,1,0.235798,0.3715,0.135702,0
4,FR,7268,1,0.248604,0.3715,0.122896,0
5,FA,6121,0,0.431952,0.3715,0.060452,1
6,FA,6465,0,0.426320,0.3715,0.054820,1
7,FA,6297,0,0.404912,0.3715,0.033412,1
8,FA,3969,0,0.392619,0.3715,0.021119,1
9,FA,709,0,0.386537,0.3715,0.015037,1



WAVLM - A


,tipo,indice,label,score,threshold,distancia_umbral,prediccion
0,FR,31178,1,0.297891,0.87354,0.575649,0
1,FR,31338,1,0.302436,0.87354,0.571104,0
2,FR,31156,1,0.318688,0.87354,0.554852,0
3,FR,31850,1,0.320693,0.87354,0.552847,0
4,FR,31180,1,0.333387,0.87354,0.540153,0
5,FA,7859,0,0.975198,0.87354,0.101658,1
6,FA,10723,0,0.972928,0.87354,0.099388,1
7,FA,28051,0,0.970113,0.87354,0.096573,1
8,FA,8303,0,0.969849,0.87354,0.096309,1
9,FA,8307,0,0.968965,0.87354,0.095425,1



WAVLM - B


,tipo,indice,label,score,threshold,distancia_umbral,prediccion
0,FR,7110,1,0.369853,0.9085,0.538647,0
1,FR,7290,1,0.649907,0.9085,0.258593,0
2,FR,7270,1,0.679365,0.9085,0.229135,0
3,FR,7268,1,0.686388,0.9085,0.222112,0
4,FR,7280,1,0.714727,0.9085,0.193773,0
5,FA,6465,0,0.982936,0.9085,0.074436,1
6,FA,6121,0,0.978155,0.9085,0.069655,1
7,FA,6307,0,0.975511,0.9085,0.067011,1
8,FA,2339,0,0.969295,0.9085,0.060795,1
9,FA,3727,0,0.968963,0.9085,0.060463,1


In [15]:
for key, df_casos in casos.items():

    protocol = key.split(" - ")[1]

    print("\n", "=" * 70)
    print(key)
    print("=" * 70)

    for _, row in df_casos.iterrows():

        if protocol == "A":
            speaker1 = Path(row["audio1"]).parts[0]
            speaker2 = Path(row["audio2"]).parts[0]

        else:
            speaker1 = row["enrollment_speaker_id"]
            speaker2 = row["test_speaker_id"]

        print(
            f"{row['tipo']} | "
            f"Score: {row['score']:.4f} | "
            f"Label: {row['label']} | "
            f"Hablante 1: {speaker1} | "
            f"Hablante 2: {speaker2}"
        )

        if row["tipo"] == "FR":
            assert speaker1 == speaker2

        elif row["tipo"] == "FA":
            assert speaker1 != speaker2

print("\nComprobación de hablantes finalizada.")


ECAPA - A
FR | Score: -0.0410 | Label: 1 | Hablante 1: id10304 | Hablante 2: id10304
FR | Score: -0.0349 | Label: 1 | Hablante 1: id10304 | Hablante 2: id10304
FR | Score: -0.0347 | Label: 1 | Hablante 1: id10304 | Hablante 2: id10304
FR | Score: -0.0042 | Label: 1 | Hablante 1: id10304 | Hablante 2: id10304
FR | Score: -0.0031 | Label: 1 | Hablante 1: id10304 | Hablante 2: id10304
FA | Score: 0.4776 | Label: 0 | Hablante 1: id10279 | Hablante 2: id10290
FA | Score: 0.4571 | Label: 0 | Hablante 1: id10300 | Hablante 2: id10278
FA | Score: 0.4338 | Label: 0 | Hablante 1: id10307 | Hablante 2: id10286
FA | Score: 0.4226 | Label: 0 | Hablante 1: id10278 | Hablante 2: id10300
FA | Score: 0.4139 | Label: 0 | Hablante 1: id10283 | Hablante 2: id10290
Borderline | Score: 0.2723 | Label: 0 | Hablante 1: id10290 | Hablante 2: id10283
Borderline | Score: 0.2721 | Label: 0 | Hablante 1: id10300 | Hablante 2: id10278
Borderline | Score: 0.2720 | Label: 0 | Hablante 1: id10296 | Hablante 2: id1030

In [16]:
def reproducir_audio(relative_path, titulo):

    path = BASE_AUDIO / str(relative_path)

    if not path.exists():
        print(f"No se encontró el archivo: {path}")
        return

    audio, sr = sf.read(path)

    print(titulo)
    display(Audio(audio.T, rate=sr))


def escuchar_caso(model, protocol, tipo, idx):

    key = f"{model} - {protocol}"

    df_casos = casos[key]

    seleccion = df_casos[
        (df_casos["tipo"] == tipo) &
        (df_casos["indice"] == idx)
    ]

    if seleccion.empty:
        print("No se encontró el caso solicitado.")
        return

    caso = seleccion.iloc[0]

    display(Markdown(
        f"""
## {key} — {tipo}

- Índice: {idx}
- Etiqueta real: {caso['label']}
- Score: {caso['score']:.6f}
- Umbral: {caso['threshold']:.6f}
- Distancia al umbral: {caso['distancia_umbral']:.6f}
- Predicción: {"Genuino" if caso['prediccion'] == 1 else "Impostor"}
        """
    ))

    if protocol == "A":

        reproducir_audio(
            caso["audio1"],
            "Audio 1"
        )

        reproducir_audio(
            caso["audio2"],
            "Audio 2"
        )

    else:

        speaker = caso["enrollment_speaker_id"]

        audios_registro = enrollment[
            enrollment["speaker_id"] == speaker
        ]["audio_path"].head(2)

        for i, audio_path in enumerate(audios_registro, 1):

            reproducir_audio(
                audio_path,
                f"Audio de registro {i} — {speaker}"
            )

        reproducir_audio(
            caso["test_audio"],
            f"Audio de prueba — {caso['test_speaker_id']}"
        )

In [17]:
# Ejemplo: peor falso rechazo de ECAPA en protocolo A
idx = int(
    casos["ECAPA - A"]
    .query("tipo == 'FR'")
    .iloc[0]["indice"]
)

escuchar_caso("ECAPA", "A", "FR", idx)


## ECAPA - A — FR

- Índice: 31156
- Etiqueta real: 1
- Score: -0.041027
- Umbral: 0.272333
- Distancia al umbral: 0.313360
- Predicción: Impostor
        

Audio 1


Audio 2


In [18]:
# Falsa aceptación de ECAPA en A
idx = int(casos["ECAPA - A"].query("tipo == 'FA'").iloc[0]["indice"])
escuchar_caso("ECAPA", "A", "FA", idx)


# Falso rechazo de WAVLM en A
idx = int(casos["WAVLM - A"].query("tipo == 'FR'").iloc[0]["indice"])
escuchar_caso("WAVLM", "A", "FR", idx)


# Falsa aceptación de WAVLM en B
idx = int(casos["WAVLM - B"].query("tipo == 'FA'").iloc[0]["indice"])
escuchar_caso("WAVLM", "B", "FA", idx)


## ECAPA - A — FA

- Índice: 8441
- Etiqueta real: 0
- Score: 0.477648
- Umbral: 0.272333
- Distancia al umbral: 0.205315
- Predicción: Genuino
        

Audio 1


Audio 2



## WAVLM - A — FR

- Índice: 31178
- Etiqueta real: 1
- Score: 0.297891
- Umbral: 0.873540
- Distancia al umbral: 0.575649
- Predicción: Impostor
        

Audio 1


Audio 2



## WAVLM - B — FA

- Índice: 6465
- Etiqueta real: 0
- Score: 0.982936
- Umbral: 0.908500
- Distancia al umbral: 0.074436
- Predicción: Genuino
        

Audio de registro 1 — id10278


Audio de registro 2 — id10278


Audio de prueba — id10300


In [19]:
# Guardar diccionario completo
with open(RESULTS_DIR / "casos_problematicos.pkl", "wb") as f:
    pickle.dump(casos, f)

# Crear tabla consolidada
tabla_casos = pd.concat(
    [
        df.assign(experimento=key)
        for key, df in casos.items()
    ],
    ignore_index=True
)

tabla_casos.to_csv(
    RESULTS_DIR / "casos_problematicos.csv",
    index=False
)

print("Casos guardados correctamente:")
print(RESULTS_DIR / "casos_problematicos.pkl")
print(RESULTS_DIR / "casos_problematicos.csv")

Casos guardados correctamente:
resultados\casos_problematicos.pkl
resultados\casos_problematicos.csv
